# Four feature views, one ensemble: LB 0.94639 from four models that disagree

My best single model here is a LightGBM at
[CV 0.94626 / LB 0.94633](https://www.kaggle.com/code/megayak/s6e9-one-lightgbm-from-raw-data-cv-0-9463).
Everything I tried *on top of it* — a full-data refit, lift-vs-original features,
centred-window target rates, max_bin 8192, additive constraints, a wider
target-encoder cross-fit, an MLP, a one-hot logistic regression, CatBoost —
came back at ±0.00003 on the same frozen ten folds. The model was not short of
features.

What worked was not adding a feature but **building the same ideas as four separate
pipelines that see the data differently, and rank-averaging them:**

| member | what differs | OOF AUC | corr. with A |
|---|---|---|---|
| A | triple target encoding (auto/10/100) + smooth keys + digit/quantised/frequency features | 0.946264 | 1 |
| B | XGBoost on exactly A's feature matrix | 0.946258 | 0.9995 |
| C | no digit features; centred-window target rates (income v±2…±200, commute v±1…±10, and the same windows within a (city, car) group); lift/novelty vs the original; smoothing 2/30/300 | 0.946223 | 0.9989 |
| D | no exact-income key; the //10, //50, //500, //5000 ladder instead; windows; smoothing auto/20/200 | 0.946077 | 0.9979 |
| **0.3 A + 0.3 B + 0.2 C + 0.2 D** | rank space | **0.946339** | |

| submission | public LB |
|---|---|
| A alone | 0.94633 |
| **A+B+C+D** | **0.94639** |
| the public 0.94644 daily blend + 30% of A+B+C+D | [0.94645](https://www.kaggle.com/code/megayak/s6e9-0-94645-four-feature-views-beat-the-blend) |

The OOF gain (+0.00006) carried to the leaderboard one-for-one, which the noise
floor I measured earlier says is not guaranteed. Two rows carry the lesson: B adds
nothing (same features, different learner, correlation 0.9995); D adds even though it
is 0.0002 weaker, because dropping the exact-value key changes what the model can
see (correlation 0.998). **Diversity comes from the feature view, not the learner.**

This notebook trains all four on the same `StratifiedKFold(10, shuffle=True,
random_state=42)` partition, in train.csv row order, and saves every OOF and test
column so you can add a fifth view and blend it row for row. It takes about two
hours on Kaggle CPU (B uses the GPU when one is usable, otherwise CPU hist).

In [ ]:
import glob, os, time, subprocess, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import rankdata, spearmanr
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
warnings.filterwarnings("ignore")

ROOT = "/kaggle/input"
NFOLD = 10
N_EST = 20000
TARGET = "Will_Buy_EV"
CATS = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible",
        "Subsidy_Available", "Range_Anxiety_Level"]
NUMS = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
        "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
        "Environmental_Concern_Level"]
rk = lambda v: rankdata(v) / len(v)

def find(name):
    hits = sorted(glob.glob(f"{ROOT}/**/{name}", recursive=True))
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]

def gpu_usable():
    try:
        subprocess.check_output("nvidia-smi", stderr=subprocess.DEVNULL, shell=True)
        import xgboost as xgb
        probe = np.random.rand(600, 4); lab = (probe[:, 0] > 0.5).astype(int)
        xgb.XGBClassifier(n_estimators=5, max_depth=3, tree_method="hist", device="cuda",
                          verbosity=0).fit(probe, lab)
        return True
    except Exception as e:
        print(f"GPU not usable for XGBoost ({type(e).__name__}); B will run on CPU")
        return False

train = pd.read_csv(find("train.csv")).drop(columns=["id"])
test = pd.read_csv(find("test.csv")); te_id = test.pop("id").to_numpy()
orig = pd.read_csv(find("EV_Adoption_and_Range_Anxiety_Dataset.csv")).drop(columns=["Buyer_ID"])
y = (train.pop(TARGET) == "Yes").astype(int).to_numpy()
tr_id = np.arange(len(train))
GPU = gpu_usable()
print(f"train {train.shape} | test {test.shape} | original {orig.shape} | gpu {GPU}")

## Features

Label-free features are built once over train+test. Everything that reads the target
(the target encodings and the window rates) is fit inside the outer fold, with an
inner 5-fold cross-fit for the training rows, so no row is ever described by a
statistic its own label went into.

In [ ]:
def build(train, test, orig, digits=True, lift=False, keys="default"):
    n = len(train)
    df = pd.concat([train, test], ignore_index=True)
    inc = df.Annual_Income_USD.to_numpy(np.int64)
    km = np.round(df.Daily_Commute_km.to_numpy(float) * 10).astype(np.int64)

    if digits:                      # the generator's low-order digit artefact
        df["inc_d1"] = (inc % 10).astype("int8"); df["inc_d2"] = (inc // 10 % 10).astype("int8")
        df["inc_d3"] = (inc // 100 % 10).astype("int8")
        df["inc_mod100"] = (inc % 100).astype("int16"); df["inc_mod1000"] = (inc % 1000).astype("int16")
        df["km_d1"] = (km % 10).astype("int8"); df["km_mod100"] = (km % 100).astype("int8")
        for d in (50, 100, 250, 500, 1000, 2500, 5000):
            df[f"inc_q{d}"] = (inc // d).astype("int32")
        for d in (5, 10, 25, 50):
            df[f"km_q{d}"] = (km // d).astype("int32")
        s = pd.Series(inc); df["fq_inc"] = s.map(s.value_counts()).astype("float32").values
        s = pd.Series(km); df["fq_km"] = s.map(s.value_counts()).astype("float32").values

    df["is_30k_spike"] = (inc == 30000).astype("int8")
    df["is_millionaire_cliff"] = (inc >= 170537).astype("int8")
    df["is_dead_zone"] = ((inc >= 38000) & (inc <= 42000)).astype("int8")
    df["is_env_hater"] = (df.Environmental_Concern_Level == 1).astype("int8")

    og = orig.dropna(subset=["Annual_Income_USD", "Daily_Commute_km", "Environmental_Concern_Level"]).copy()
    og[TARGET] = (og[TARGET] == "Yes").astype(int); gm = og[TARGET].mean()
    for c in CATS + NUMS:
        df[f"{c}_org_mean"] = df[c].map(og.groupby(c)[TARGET].mean()).fillna(gm).astype("float32")

    if lift:                        # how much the generator over-produced each value vs the original
        oi = np.round(orig.Annual_Income_USD.dropna().to_numpy(float)).astype(np.int64)
        ok = np.round(orig.Daily_Commute_km.dropna().to_numpy(float) * 10).astype(np.int64)
        for name, comp, src, divs in (("inc", inc, oi, (1, 100, 1000)), ("km", km, ok, (1, 10))):
            for d in divs:
                c_s = pd.Series(comp // d); o_s = pd.Series(src // d)
                c_cnt = c_s.map(c_s.value_counts()).to_numpy(float)
                o_cnt = c_s.map(o_s.value_counts()).fillna(0).to_numpy(float)
                sfx = "" if d == 1 else f"_q{d}"
                df[f"lift_{name}{sfx}"] = (np.log((c_cnt + 1) / len(df)) - np.log((o_cnt + 1) / len(src))).astype("float32")
                df[f"ocnt_{name}{sfx}"] = o_cnt.astype("float32")
                if d == 1:
                    df[f"novel_{name}"] = (o_cnt == 0).astype("int8")

    te = {}
    if keys == "alt":
        for d in (10, 50, 500, 5000): te[f"k_inc{d}"] = (inc // d).astype(str)
        for d in (5, 50): te[f"k_km{d}"] = (km // d).astype(str)
    else:
        te["k_inc_exact"] = inc.astype(str); te["k_inc100"] = (inc // 100).astype(str)
        te["k_inc1000"] = (inc // 1000).astype(str); te["k_km_int"] = (km // 10).astype(str)
    for c in CATS:
        te[f"k_{c}"] = df[c].astype(str).to_numpy()
    for c in ("Age", "Number_of_Cars_Owned", "Charging_Stations_Near_Home",
              "Charging_Stations_Near_Work", "Environmental_Concern_Level"):
        te[f"k_{c}"] = df[c].astype(str).to_numpy()
    K = pd.DataFrame(te)
    for c in K.columns:
        df[f"{c}_fe"] = K[c].map(K[c].value_counts(normalize=True)).astype("float32").values
    for c in CATS:
        df[c] = df[c].astype("category")
    return (df.iloc[:n].reset_index(drop=True), df.iloc[n:].reset_index(drop=True),
            K.iloc[:n].reset_index(drop=True), K.iloc[n:].reset_index(drop=True))


WIN_INC, WIN_KM, WIN_GRP, WIN_PRIOR = (2, 5, 10, 25, 50, 200), (1, 3, 10), (5, 25), 10.0
WIN_COLS = ([f"win_inc_{w}" for w in WIN_INC] + [f"win_km_{w}" for w in WIN_KM]
            + [f"win_inc_{w}_citycar" for w in WIN_GRP])

def window_rates(v_fit, y_fit, v_query, widths, prior, gm):
    # smoothed buy rate over the centred window [v-w, v+w], from (v_fit, y_fit) only
    lo, hi = int(min(v_fit.min(), v_query.min())), int(max(v_fit.max(), v_query.max()))
    n = hi - lo + 1
    cnt = np.bincount(v_fit - lo, minlength=n).astype(np.float64)
    s = np.bincount(v_fit - lo, weights=y_fit, minlength=n)
    ccnt = np.concatenate([[0.0], np.cumsum(cnt)]); cs = np.concatenate([[0.0], np.cumsum(s)])
    q = v_query - lo; out = []
    for w in widths:
        a = np.clip(q - w, 0, n); b = np.clip(q + w + 1, 0, n)
        out.append((((cs[b] - cs[a]) + prior * gm) / ((ccnt[b] - ccnt[a]) + prior)).astype("float32"))
    return np.stack(out, 1)

def window_block(inc_a, km_a, y_a, inc_q, km_q, g_a, g_q):
    gm = float(y_a.mean())
    parts = [window_rates(inc_a, y_a, inc_q, WIN_INC, WIN_PRIOR, gm),
             window_rates(km_a, y_a, km_q, WIN_KM, WIN_PRIOR, gm)]
    out = np.full((len(inc_q), len(WIN_GRP)), gm, np.float32)
    for g in np.unique(g_q):
        ma, mq = g_a == g, g_q == g
        if ma.sum():
            out[mq] = window_rates(inc_a[ma], y_a[ma], inc_q[mq], WIN_GRP, WIN_PRIOR, gm)
    parts.append(out)
    return np.concatenate(parts, 1)

## One fold loop, four configurations

In [ ]:
LGB = dict(n_estimators=N_EST, learning_rate=0.02, max_depth=5, num_leaves=32, min_child_samples=10,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.3, reg_alpha=0.071, reg_lambda=2.0,
           max_bin=1024, feature_pre_filter=False, n_jobs=-1, verbose=-1)

def run_view(name, digits=True, lift=False, keys="default", win=False,
             smooth=("auto", 10.0, 100.0), model="lgb", seed=42):
    X, Xte, K, Kte = build(train, test, orig, digits=digits, lift=lift, keys=keys)
    inc_all = np.concatenate([X.Annual_Income_USD.to_numpy(np.int64), Xte.Annual_Income_USD.to_numpy(np.int64)])
    km_all = np.round(np.concatenate([X.Daily_Commute_km.to_numpy(float), Xte.Daily_Commute_km.to_numpy(float)]) * 10).astype(np.int64)
    g_all = pd.factorize(pd.concat([X.City_Type.astype(str) + "|" + X.Current_Car_Type.astype(str),
                                    Xte.City_Type.astype(str) + "|" + Xte.Current_Car_Type.astype(str)],
                                   ignore_index=True))[0]
    n = len(X); inc_tr, km_tr, g_tr = inc_all[:n], km_all[:n], g_all[:n]
    inc_te, km_te, g_te = inc_all[n:], km_all[n:], g_all[n:]

    cv = StratifiedKFold(NFOLD, shuffle=True, random_state=42)
    oof = np.zeros(n); pte = np.zeros(len(Xte)); its = []; t0 = time.time()
    print(f"\n=== view {name}: base features {X.shape[1]}, TE keys {K.shape[1]} x {len(smooth)}, model {model} ===")
    for f, (a, b) in enumerate(cv.split(X, y)):
        A, B, C = X.iloc[a].copy(), X.iloc[b].copy(), Xte.copy()
        for sm in smooth:
            tag = "auto" if sm == "auto" else str(int(sm))
            enc = TargetEncoder(shuffle=True, cv=5, smooth=sm, random_state=42)
            ea = enc.fit_transform(K.iloc[a], y[a]); eb = enc.transform(K.iloc[b]); ec = enc.transform(Kte)
            for i, c in enumerate(K.columns):
                A[f"{c}_te{tag}"] = ea[:, i].astype("float32"); B[f"{c}_te{tag}"] = eb[:, i].astype("float32")
                C[f"{c}_te{tag}"] = ec[:, i].astype("float32")
        if win:
            wa = np.zeros((len(a), len(WIN_COLS)), np.float32)
            for ia, ib in StratifiedKFold(5, shuffle=True, random_state=42).split(a, y[a]):
                wa[ib] = window_block(inc_tr[a[ia]], km_tr[a[ia]], y[a[ia]].astype(float),
                                      inc_tr[a[ib]], km_tr[a[ib]], g_tr[a[ia]], g_tr[a[ib]])
            wb = window_block(inc_tr[a], km_tr[a], y[a].astype(float), inc_tr[b], km_tr[b], g_tr[a], g_tr[b])
            wc = window_block(inc_tr[a], km_tr[a], y[a].astype(float), inc_te, km_te, g_tr[a], g_te)
            for i, c in enumerate(WIN_COLS):
                A[c] = wa[:, i]; B[c] = wb[:, i]; C[c] = wc[:, i]
        if model == "xgb":
            import xgboost as xgb
            m = xgb.XGBClassifier(n_estimators=N_EST, learning_rate=0.02, max_depth=5, min_child_weight=5,
                                  subsample=0.8, colsample_bytree=0.3, reg_alpha=0.071, reg_lambda=2.0,
                                  max_bin=1024, tree_method="hist", device="cuda" if GPU else "cpu",
                                  enable_categorical=True, eval_metric="auc", early_stopping_rounds=500,
                                  random_state=seed, n_jobs=-1, verbosity=0)
            m.fit(A, y[a], eval_set=[(B, y[b])], verbose=False); it = int(m.best_iteration)
        else:
            m = lgb.LGBMClassifier(random_state=seed, **LGB)
            m.fit(A, y[a], eval_set=[(B, y[b])], eval_metric="auc",
                  callbacks=[lgb.early_stopping(500, verbose=False)]); it = int(m.best_iteration_)
        pb = m.predict_proba(B)[:, 1]
        oof[b] = rk(pb); pte += rk(m.predict_proba(C)[:, 1]) / NFOLD; its.append(it)
        print(f"  fold{f} auc={roc_auc_score(y[b], pb):.5f} it={it} ncol={A.shape[1]} [{time.time()-t0:.0f}s]", flush=True)
    auc = roc_auc_score(y, oof)
    print(f"  {name} OOF AUC = {auc:.6f}   iters {its}")
    return oof, rk(pte), auc

## Train the four views

In [ ]:
VIEWS = {
    "A": dict(digits=True,  lift=False, keys="default", win=False, smooth=("auto", 10.0, 100.0), model="lgb"),
    "B": dict(digits=True,  lift=False, keys="default", win=False, smooth=("auto", 10.0, 100.0), model="xgb"),
    "C": dict(digits=False, lift=True,  keys="default", win=True,  smooth=(2.0, 30.0, 300.0),    model="lgb"),
    "D": dict(digits=True,  lift=False, keys="alt",     win=True,  smooth=("auto", 20.0, 200.0), model="lgb"),
}
WEIGHTS = {"A": 0.3, "B": 0.3, "C": 0.2, "D": 0.2}

OOF, TEST, AUC = {}, {}, {}
for name, cfg in VIEWS.items():
    OOF[name], TEST[name], AUC[name] = run_view(name, **cfg)

oof_df = pd.DataFrame({"row": tr_id, **{f"view_{k}": v.astype("float32") for k, v in OOF.items()}})
test_df = pd.DataFrame({"id": te_id, **{f"view_{k}": v.astype("float32") for k, v in TEST.items()}})

ens_oof = rk(sum(WEIGHTS[k] * OOF[k] for k in VIEWS))
ens_test = rk(sum(WEIGHTS[k] * TEST[k] for k in VIEWS))
oof_df["ensemble"] = ens_oof.astype("float32"); test_df["ensemble"] = ens_test.astype("float32")
oof_df.to_csv("oof_four_views.csv", index=False); test_df.to_csv("test_four_views.csv", index=False)
pd.DataFrame({"id": te_id, TARGET: ens_test}).to_csv("submission.csv", index=False)

print("\nmember OOF AUC:", {k: round(v, 6) for k, v in AUC.items()})
print(f"ensemble OOF AUC = {roc_auc_score(y, ens_oof):.6f}   (equal weights {roc_auc_score(y, rk(sum(OOF.values()))):.6f})")
print("\npairwise OOF spearman vs A:")
for k in ("B", "C", "D"):
    print(f"  A ~ {k}: {spearmanr(OOF['A'], OOF[k]).statistic:.5f}")
print("\nwrote submission.csv, oof_four_views.csv, test_four_views.csv")

## What I would try next

The pace is about +0.00003 OOF per additional view, and this time it carried to the
board in full. The obvious fifth and sixth members are a different quantisation ladder
(//25, //250, //2500) and a model that takes the per-value target rate as
`init_score` instead of as columns — the forum's "everything we measured" thread found
that formulation to be the most decorrelated member anyone built. If you add a view,
the `oof_four_views.csv` here is on the same folds, so you can blend without refitting.

Things that did *not* add a view, measured on these folds: a full-data refit
(+0.00001 LB), lift/novelty features alone (+0.000015), window rates alone
(−0.00001), max_bin 8192 (0), `interaction_constraints` (−0.0004), a wider
target-encoder cross-fit (−0.00007), an MLP on the same features (−0.0010, and
negative in the blend), a one-hot logistic (−0.003), CatBoost with native categorical
keys (−0.0005). Same ideas, one model: nothing. Same ideas, four models: +0.00006.

---

Credit: the target-encoding recipe follows [@najiama](https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94606-lb-0-94637);
the original dataset is [@itzzomkar](https://www.kaggle.com/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety);
the window-rate idea comes from the forum discussion started by @starkhushi and @tilii7.
The four views, the ensemble and the measurements are mine. If this saved you a
day of single-model tweaks, an upvote helps.